## Q2(f) — Reconciliation and Model Risk

### 1. Market Quote and Half-Spread Normalization

The market midquote is calculated as the average of the bid and ask prices. The market half-spread is half of the bid–ask spread.

For each pricing method, the signed distance from the market midquote is defined as:

D ={C_model-C_market_mid}/
(ask-bid)/2

C_market_mid = (ask+bid)/2

A positive value indicates that the model price is above the market midquote, while a negative value indicates that it is below. A value between -1 and 1 indicates that the model price lies within the quoted bid–ask spread.

In [19]:
# Q2(f): Market quote normalization

# Replace these placeholders with the actual Q1 option bid and ask.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from time import perf_counter

s0 = 209.05
K = 210
r = 0.045
sigma = 0.497988  # Replace with full-precision Q1 call IV

valuation_date = pd.Timestamp("2026-10-09")
expiration_date = pd.Timestamp("2027-01-15")

T = (expiration_date-valuation_date).days/365
print("Time to maturity: ", T)

bs_call = 22.175019659856545 #from Q1

market_bid = 22
market_ask = 22.35

market_mid = (market_bid+market_ask)/2
half_spread = (market_ask-market_bid)/2

def signed_distance(model_price, bid, ask):
    if ask <= bid:
        raise ValueError("Ask must be greater than Bid")

    mid = (bid+ask)/2
    half = (ask-bid)/2
    return(model_price-mid)/half

print("Market Midquote:", market_mid)
print("Market Half-Spread:", half_spread)
print("BS Signed Distance:",
      signed_distance(22.17501966, market_bid, market_ask))

Time to maturity:  0.2684931506849315
Market Midquote: 22.175
Market Half-Spread: 0.1750000000000007
BS Signed Distance: 0.00011234285714018713


### 2. Reconciliation of Option Pricing Methods

This section compares the Black–Scholes analytical benchmark, Monte Carlo simulation, binomial tree methods, finite-difference methods, and the working risk-neutral GARCH scenario.

All methods price the same PLTR call option with a strike price of $210 and an expiration date of January 15, 2027.

The comparison includes the volatility specification, estimated option price, numerical error or confidence interval, computational runtime, and signed distance from the market midquote in units of the market half-spread.

In [5]:
methods = [
    "Black-Scholes",
    "Crude Monte Carlo",
    "CRR Binomial Tree",
    "Jarrow-Rudd Binomial Tree",
    "Explicit FDM",
    "Fully Implicit FDM",
    "Crank-Nicolson FDM",
    "GARCH Monte Carlo"
]

volatility_specs = [
    "Constant implied volatility",
    "Constant implied volatility",
    "Constant implied volatility",
    "Constant implied volatility",
    "Constant implied volatility",
    "Constant implied volatility",
    "Constant implied volatility",
    "Stochastic GARCH(1,1)"
]

reconciliation_df = pd.DataFrame({
    "Method": methods,
    "Volatility": volatility_specs,
    "Price": np.nan,
    "CI / Deterministic Error": None,
    "Runtime (s)": np.nan,
    "Signed Distance": np.nan
})

display(reconciliation_df)

,Method,Volatility,Price,CI / Deterministic Error,Runtime (s),Signed Distance
0,Black-Scholes,Constant implied volatility,NaN,None,NaN,NaN
1,Crude Monte Carlo,Constant implied volatility,NaN,None,NaN,NaN
2,CRR Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
3,Jarrow-Rudd Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
4,Explicit FDM,Constant implied volatility,NaN,None,NaN,NaN
5,Fully Implicit FDM,Constant implied volatility,NaN,None,NaN,NaN
6,Crank-Nicolson FDM,Constant implied volatility,NaN,None,NaN,NaN
7,GARCH Monte Carlo,"Stochastic GARCH(1,1)",NaN,None,NaN,NaN


In [6]:
reconciliation_df.loc[
    reconciliation_df["Method"] == "Black-Scholes",
    "Price"
] = bs_call

reconciliation_df["Signed Distance"] = reconciliation_df["Price"].apply(
    lambda price: signed_distance(price, market_bid, market_ask)
    if pd.notna(price) else np.nan
)

display(reconciliation_df)

,Method,Volatility,Price,CI / Deterministic Error,Runtime (s),Signed Distance
0,Black-Scholes,Constant implied volatility,22.17502,None,NaN,0.000112
1,Crude Monte Carlo,Constant implied volatility,NaN,None,NaN,NaN
2,CRR Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
3,Jarrow-Rudd Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
4,Explicit FDM,Constant implied volatility,NaN,None,NaN,NaN
5,Fully Implicit FDM,Constant implied volatility,NaN,None,NaN,NaN
6,Crank-Nicolson FDM,Constant implied volatility,NaN,None,NaN,NaN
7,GARCH Monte Carlo,"Stochastic GARCH(1,1)",NaN,None,NaN,NaN


### 4. Populating the Reconciliation Table

The results from the individual pricing notebooks are consolidated into a common reconciliation table. For numerical methods under the constant-volatility assumption, the deterministic pricing error is measured against the Black–Scholes analytical benchmark.

For Monte Carlo methods, the reported uncertainty is the 95% confidence interval. The GARCH price is treated as an alternative model result rather than a numerical approximation to Black–Scholes.

In [7]:
def add_model_result(method, price, runtime, ci_or_error):
    mask = reconciliation_df["Method"] ==method

    if not mask.any():
        raise ValueError(f"Unknwon methd: {method}")
    reconciliation_df.loc[mask, "Price"] = price
    reconciliation_df.loc[mask, "Runtime (s)"] = runtime
    reconciliation_df.loc[mask, "CI / Deterministic Error"] = ci_or_error

    reconciliation_df.loc[mask, "Signed Distance"] = (
        signed_distance(price, market_bid, market_ask)
    )

### 5. Crude Monte Carlo Reconciliation

The crude Monte Carlo benchmark uses 262,144 simulated paths. The estimated call price is $22.191460, with a 95% confidence interval of [$22.048461, $22.334460].

The Black–Scholes analytical benchmark lies within this confidence interval. The signed distance from the market midquote is approximately +0.094 half-spreads, indicating that the Monte Carlo estimate lies within the observed market bid–ask spread.

The difference between the Monte Carlo estimate and the analytical benchmark is consistent with finite-sample simulation uncertainty.

In [8]:
mc_price = 22.191460
mc_ci_lower = 22.048461
mc_ci_upper = 22.334460
mc_runtime = 0.006486

add_model_result(
    method="Crude Monte Carlo",
    price=mc_price,
    runtime=mc_runtime,
    ci_or_error=f"95% CI [{mc_ci_lower:.6f}, {mc_ci_upper:.6f}]"
)

display(reconciliation_df)

,Method,Volatility,Price,CI / Deterministic Error,Runtime (s),Signed Distance
0,Black-Scholes,Constant implied volatility,22.17502,None,NaN,0.000112
1,Crude Monte Carlo,Constant implied volatility,22.19146,"95% CI [22.048461, 22.334460]",0.006486,0.094057
2,CRR Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
3,Jarrow-Rudd Binomial Tree,Constant implied volatility,NaN,None,NaN,NaN
4,Explicit FDM,Constant implied volatility,NaN,None,NaN,NaN
5,Fully Implicit FDM,Constant implied volatility,NaN,None,NaN,NaN
6,Crank-Nicolson FDM,Constant implied volatility,NaN,None,NaN,NaN
7,GARCH Monte Carlo,"Stochastic GARCH(1,1)",NaN,None,NaN,NaN


### CRR Binomial Tree Reconciliation

The CRR binomial tree was evaluated over several time-step refinements. Among the tested grids, the 400-step tree achieved the smallest absolute pricing error relative to the Black–Scholes analytical benchmark.

At 400 steps, the CRR price is $22.177026, with an absolute deterministic error of $0.002006 and a runtime of approximately 0.002417 seconds. Its signed distance from the market midquote is +0.0116 half-spreads.

The non-monotonic pricing errors across grid refinements indicate that a larger number of time steps does not necessarily produce a smaller error for every individual grid.

Importing Binomial Tree Results
The previously computed CRR and JRR results are loaded to ensure consistent pricing inputs and avoid repeating the numerical calculations.

In [10]:
import pandas as pd

crr_df = pd.read_csv("../Data/processed/crr_results.csv")
jrr_df = pd.read_csv("../Data/processed/jrr_results.csv")

display(crr_df)
display(jrr_df)

,N,Price,Delta,Gamma,Probability,Runtime (s),Price Error,Delta Error,Gamma Error,Admissible
0,25,22.383284,0.562751,0.007372,0.491782,0.000300,0.208265,0.000128,0.000068,True
1,50,22.117601,0.562641,0.007396,0.494189,0.000408,0.057419,0.000238,0.000092,True
2,100,22.155570,0.562774,0.007346,0.495890,0.000694,0.019450,0.000105,0.000042,True
3,200,22.171380,0.562835,0.007322,0.497094,0.001051,0.003639,0.000043,0.000019,True
4,400,22.177026,0.562862,0.007312,0.497945,0.002113,0.002006,0.000016,0.000008,True
5,800,22.178248,0.562874,0.007307,0.498547,0.006110,0.003229,0.000005,0.000003,True


,N,Price,Delta,Gamma,Probability,Runtime (s),Price Error,Delta Error,Gamma Error,Admissible
0,25,22.272315,0.561345,0.007433,0.5,0.000230,0.097295,0.001533,0.000129,True
1,50,22.258695,0.562479,0.007354,0.5,0.000229,0.083676,0.000400,0.000051,True
2,100,22.225969,0.562646,0.007326,0.5,0.000433,0.050949,0.000233,0.000022,True
3,200,22.191472,0.562730,0.007318,0.5,0.000774,0.016452,0.000149,0.000014,True
4,400,22.161425,0.562772,0.007318,0.5,0.001518,0.013594,0.000107,0.000015,True
5,800,22.180961,0.562850,0.007306,0.5,0.004790,0.005942,0.000029,0.000003,True


In [11]:
crr_row = crr_df.loc[crr_df["N"] == 400].iloc[0]

crr_price = crr_row["Price"]
crr_runtime = crr_row["Runtime (s)"]
crr_error = crr_row["Price Error"]

print("CRR Price:", crr_price)
print("Deterministic Error:", crr_error)
print("Runtime:", crr_runtime)
print(
    "Signed Distance:",
    signed_distance(crr_price, market_bid, market_ask)
)

CRR Price: 22.17702601845321
Deterministic Error: 0.0020063585966632
Runtime: 0.0021128000080352
Signed Distance: 0.011577248304044021


### JRR Binomial Tree Reconciliation

The Jarrow–Rudd binomial tree is evaluated using 800 time steps, which produces the smallest absolute pricing error among the tested refinements. The resulting price, deterministic error relative to Black–Scholes, runtime, and signed distance from the market midquote are extracted for the final reconciliation table.

In [12]:
jrr_row = jrr_df.loc[jrr_df["N"] == 800].iloc[0]

jrr_price = jrr_row["Price"]
jrr_runtime = jrr_row["Runtime (s)"]
jrr_error = jrr_row["Price Error"]

print("JRR Price:", jrr_price)
print("Deterministic Error:", jrr_error)
print("Runtime:", jrr_runtime)
print(
    "Signed Distance:",
    signed_distance(jrr_price, market_bid, market_ask)
)

JRR Price: 22.18096119259892
Deterministic Error: 0.0059415327423764
Runtime: 0.0047896000032778
Signed Distance: 0.03406395770811939


### Importing Monte Carlo Results

The Monte Carlo convergence results are loaded from the previous experiment. The estimate with the largest number of simulation paths is selected, and its sampling uncertainty is represented by the 95% confidence interval.

In [13]:
mc_df = pd.read_csv("../Data/processed/mc_results.csv")

mc_row = mc_df.loc[mc_df["N"].idxmax()]

mc_price = mc_row["MC price"]
mc_runtime = mc_row["Run time(s)"]
mc_ci_lower = mc_row["CI Lower"]
mc_ci_upper = mc_row["CI Upper"]

print("MC Price:", mc_price)
print("95% CI:", (mc_ci_lower, mc_ci_upper))
print("Runtime:", mc_runtime)
print(
    "Signed Distance:",
    signed_distance(mc_price, market_bid, market_ask)
)

MC Price: 22.194452166007576
95% CI: (np.float64(22.051454691037623), np.float64(22.33744964097753))
Runtime: 0.0095185999962268
Signed Distance: 0.1111552343289998


### Loading Pricing Results for Reconciliation

The pricing results from the Monte Carlo, binomial tree, finite-difference, and GARCH simulations are imported from previously generated CSV files. This ensures that all numerical methods are compared using their saved outputs without rerunning the simulations.

In [14]:
from pathlib import Path
import pandas as pd

data_dir = Path("../Data/processed")

for file in sorted(data_dir.glob("*.csv")):
    print(file.name)

crr_results.csv
ewma_result.csv
explicit_fdm_results.csv
fdm_reconciliation.csv
garch_result.csv
gbm_result.csv
jrr_results.csv
mc_results.csv


### Inspecting Imported Pricing Results

Before constructing the reconciliation table, the column names and sample records of each exported dataset are inspected. This step ensures that pricing estimates, confidence intervals, deterministic errors, and runtimes are extracted consistently across numerical methods.

In [15]:
files_to_check = [
    "crr_results.csv",
    "jrr_results.csv",
    "mc_results.csv",
    "explicit_fdm_results.csv",
    "fdm_reconciliation.csv",
    "garch_result.csv",
    "gbm_result.csv",
    "ewma_result.csv"
]

for filename in files_to_check:
    df = pd.read_csv(data_dir / filename)

    print(f"\n--- {filename} ---")
    print("Columns:", df.columns.tolist())
    display(df.head(3))


--- crr_results.csv ---
Columns: ['N', 'Price', 'Delta', 'Gamma', 'Probability', 'Runtime (s)', 'Price Error', 'Delta Error', 'Gamma Error', 'Admissible']


,N,Price,Delta,Gamma,Probability,Runtime (s),Price Error,Delta Error,Gamma Error,Admissible
0,25,22.383284,0.562751,0.007372,0.491782,0.000300,0.208265,0.000128,0.000068,True
1,50,22.117601,0.562641,0.007396,0.494189,0.000408,0.057419,0.000238,0.000092,True
2,100,22.155570,0.562774,0.007346,0.495890,0.000694,0.019450,0.000105,0.000042,True



--- jrr_results.csv ---
Columns: ['N', 'Price', 'Delta', 'Gamma', 'Probability', 'Runtime (s)', 'Price Error', 'Delta Error', 'Gamma Error', 'Admissible']


,N,Price,Delta,Gamma,Probability,Runtime (s),Price Error,Delta Error,Gamma Error,Admissible
0,25,22.272315,0.561345,0.007433,0.5,0.000230,0.097295,0.001533,0.000129,True
1,50,22.258695,0.562479,0.007354,0.5,0.000229,0.083676,0.000400,0.000051,True
2,100,22.225969,0.562646,0.007326,0.5,0.000433,0.050949,0.000233,0.000022,True



--- mc_results.csv ---
Columns: ['N', 'MC price', 'Standard error', 'CI Lower', 'CI Upper', 'BS Call', 'Absolute error', 'Run time(s)']


...

,forecast_date,omega,alpha,beta,mu,next_daily_variance,annualized_volatility
0,2026-10-09,3.510181,0.048707,0.756854,0.19626,0.001485,0.611688



--- gbm_result.csv ---
Columns: ['model', 'annualized_volatility']


,model,annualized_volatility
0,Constant_GBM,0.672856



--- ewma_result.csv ---
Columns: ['forecast_date', 'lambda', 'daily_variance', 'annualized_volatility']


,forecast_date,lambda,daily_variance,annualized_volatility
0,2026-10-09,0.97,0.001268,0.565345


### Reconciliation of Numerical Option Pricing Methods

The Black–Scholes analytical price is used as the benchmark for numerical pricing errors. CRR, JRR, and finite-difference methods report absolute deterministic deviations from the analytical benchmark, while Monte Carlo reports a 95% confidence interval.

All model prices are also compared with the observed market midquote. The signed distance is normalized by the market half-spread, allowing discrepancies to be interpreted relative to the quoted bid–ask spread.

In [16]:
crr_df = pd.read_csv(data_dir / "crr_results.csv")
jrr_df = pd.read_csv(data_dir / "jrr_results.csv")
mc_df = pd.read_csv(data_dir / "mc_results.csv")
fdm_df = pd.read_csv(data_dir / "fdm_reconciliation.csv")

# Select representative numerical resolutions
crr_row = crr_df.loc[crr_df["N"] == 400].iloc[0]
jrr_row = jrr_df.loc[jrr_df["N"] == 400].iloc[0]
mc_row = mc_df.loc[mc_df["N"].idxmax()]

reconciliation_rows = [
    {
        "Method": "Black-Scholes",
        "Volatility Specification": "Market implied volatility",
        "Price": bs_call,
        "95% CI / Deterministic Error": "Analytical benchmark",
        "Runtime (s)": None,
    },
    {
        "Method": "CRR Binomial",
        "Volatility Specification": "Market implied volatility",
        "Price": crr_row["Price"],
        "95% CI / Deterministic Error": f'{crr_row["Price Error"]:.6f}',
        "Runtime (s)": crr_row["Runtime (s)"],
    },
    {
        "Method": "JRR Binomial",
        "Volatility Specification": "Market implied volatility",
        "Price": jrr_row["Price"],
        "95% CI / Deterministic Error": f'{jrr_row["Price Error"]:.6f}',
        "Runtime (s)": jrr_row["Runtime (s)"],
    },
    {
        "Method": "Monte Carlo",
        "Volatility Specification": "Market implied volatility",
        "Price": mc_row["MC price"],
        "95% CI / Deterministic Error":
            f'[{mc_row["CI Lower"]:.4f}, {mc_row["CI Upper"]:.4f}]',
        "Runtime (s)": mc_row["Run time(s)"],
    },
]

for _, row in fdm_df.iterrows():
    reconciliation_rows.append({
        "Method": row["Method"],
        "Volatility Specification": "Market implied volatility",
        "Price": row["Price"],
        "95% CI / Deterministic Error": f'{row["Price Error"]:.6f}',
        "Runtime (s)": row["Runtime (s)"],
    })

reconciliation_df = pd.DataFrame(reconciliation_rows)

reconciliation_df["Signed Distance (Half-Spreads)"] = (
    reconciliation_df["Price"].apply(
        lambda price: signed_distance(price, market_bid, market_ask)
    )
)

display(reconciliation_df)

,Method,Volatility Specification,Price,95% CI / Deterministic Error,Runtime (s),Signed Distance (Half-Spreads)
0,Black-Scholes,Market implied volatility,22.175020,Analytical benchmark,NaN,0.000112
1,CRR Binomial,Market implied volatility,22.177026,0.002006,0.002113,0.011577
2,JRR Binomial,Market implied volatility,22.161425,0.013594,0.001518,-0.077570
3,Monte Carlo,Market implied volatility,22.194452,"[22.0515, 22.3374]",0.009519,0.111155
4,Explicit FDM,Market implied volatility,22.175143,0.000123,0.024103,0.000817
5,Fully Implicit FDM,Market implied volatility,22.174623,0.000397,0.767151,-0.002154
6,Crank-Nicolson FDM,Market implied volatility,22.175025,0.000005,0.803265,0.000143


### Incorporating GARCH Monte Carlo into the Reconciliation

The GARCH Monte Carlo estimate is added to the pricing comparison using the simulation with the largest number of paths. Its confidence interval captures Monte Carlo sampling uncertainty, while its volatility specification reflects the dynamics of the GARCH(1,1) model.

In [17]:
garch_mc_df = pd.read_csv(
    data_dir / "garch_mc_results.csv"
)

garch_row = garch_mc_df.loc[
    garch_mc_df["N Paths"].idxmax()
]

garch_entry = {
    "Method": "GARCH Monte Carlo",
    "Volatility Specification": "IV-anchored GARCH(1,1)",
    "Price": garch_row["Price"],
    "95% CI / Deterministic Error": (
        f'[{garch_row["CI Lower"]:.4f}, '
        f'{garch_row["CI Upper"]:.4f}]'
    ),
    "Runtime (s)": garch_row["Runtime (s)"],
    "Signed Distance (Half-Spreads)": signed_distance(
        garch_row["Price"],
        market_bid,
        market_ask
    )
}

reconciliation_df = reconciliation_df[
    reconciliation_df["Method"] != "GARCH Monte Carlo"
].copy()

reconciliation_df = pd.concat(
    [reconciliation_df, pd.DataFrame([garch_entry])],
    ignore_index=True
)

display(reconciliation_df)

,Method,Volatility Specification,Price,95% CI / Deterministic Error,Runtime (s),Signed Distance (Half-Spreads)
0,Black-Scholes,Market implied volatility,22.175020,Analytical benchmark,NaN,0.000112
1,CRR Binomial,Market implied volatility,22.177026,0.002006,0.002113,0.011577
2,JRR Binomial,Market implied volatility,22.161425,0.013594,0.001518,-0.077570
3,Monte Carlo,Market implied volatility,22.194452,"[22.0515, 22.3374]",0.009519,0.111155
4,Explicit FDM,Market implied volatility,22.175143,0.000123,0.024103,0.000817
5,Fully Implicit FDM,Market implied volatility,22.174623,0.000397,0.767151,-0.002154
6,Crank-Nicolson FDM,Market implied volatility,22.175025,0.000005,0.803265,0.000143
7,GARCH Monte Carlo,"IV-anchored GARCH(1,1)",22.750521,"[22.5814, 22.9197]",0.481722,3.288692


### Input Uncertainty: Implied Volatility Sensitivity

To quantify input uncertainty, the Black–Scholes call price is recalculated under alternative implied volatility assumptions. A symmetric perturbation of ±1 volatility percentage point is used as a sensitivity scenario, rather than as an estimated confidence interval for implied volatility.

The resulting price changes are compared with the observed market half-spread to assess whether modest volatility input changes can produce economically material pricing differences.

In [20]:
from scipy.stats import norm

def bs_call_with_sigma(vol):
    d1 = (
        np.log(s0 / K)
        + (r + 0.5 * vol**2) * T
    ) / (vol * np.sqrt(T))

    d2 = d1 - vol * np.sqrt(T)

    return (
        s0 * norm.cdf(d1)
        - K * np.exp(-r * T) * norm.cdf(d2)
    )


vol_scenarios = [
    sigma - 0.01,
    sigma,
    sigma + 0.01
]

sensitivity_results = []

for vol in vol_scenarios:
    price = bs_call_with_sigma(vol)

    sensitivity_results.append({
        "Implied Volatility": vol,
        "BS Price": price,
        "Price Change vs Base": price - bs_call,
        "Change in Half-Spreads": (
            (price - bs_call) / half_spread
        )
    })

iv_sensitivity_df = pd.DataFrame(sensitivity_results)

display(iv_sensitivity_df)

,Implied Volatility,BS Price,Price Change vs Base,Change in Half-Spreads
0,0.487988,21.748189,-0.426831,-2.439032
1,0.497988,22.175020,0.000000,0.000000
2,0.507988,22.601715,0.426695,2.438259


### Sampling Error: Monte Carlo Statistical Consistency

Monte Carlo sampling uncertainty is assessed by standardizing the difference between the simulated option price and the analytical Black–Scholes benchmark using the estimated Monte Carlo standard error.

The resulting standardized discrepancy is compared with the conventional 95% normal threshold of 1.96. This provides a numerical check of whether the observed pricing difference is consistent with sampling variability under the same pricing model.

In [21]:
mc_standard_error = mc_row["Standard error"]

mc_z_score = (mc_price - bs_call) / mc_standard_error

print(f"MC Price: {mc_price:.6f}")
print(f"BS Price: {bs_call:.6f}")
print(f"MC Standard Error: {mc_standard_error:.6f}")
print(f"Standardized Difference: {mc_z_score:.4f}")
print(f"Within 95% threshold: {abs(mc_z_score) <= 1.96}")

MC Price: 22.194452
BS Price: 22.175020
MC Standard Error: 0.072958
Standardized Difference: 0.2664
Within 95% threshold: True


### Extracting GARCH Monte Carlo Results

The GARCH Monte Carlo price and confidence interval are retrieved from the previously constructed reconciliation table. This avoids rerunning the simulation and ensures that the statistical discrepancy analysis uses the same results reported in the final comparison.

In [22]:
garch_row = reconciliation_df.loc[
    reconciliation_df["Method"] == "GARCH Monte Carlo"
].iloc[0]

garch_price = float(garch_row["Price"])

# The confidence interval is stored as a formatted string.
ci_text = garch_row["95% CI / Deterministic Error"]

garch_ci_lower, garch_ci_upper = [
    float(x.strip())
    for x in ci_text.strip("[]").split(",")
]

print("GARCH Price:", garch_price)
print("GARCH 95% CI:", garch_ci_lower, garch_ci_upper)

GARCH Price: 22.750521111678648
GARCH 95% CI: 22.5814 22.9197


### GARCH Monte Carlo Sampling Error Check

The standardized difference between the GARCH Monte Carlo price and the Black–Scholes benchmark is calculated using the GARCH simulation standard error inferred from its reported 95% confidence interval.

In [23]:
garch_standard_error = (
    garch_ci_upper - garch_ci_lower
) / (2 * 1.96)

garch_z_score = (
    garch_price - bs_call
) / garch_standard_error

print(f"GARCH Standard Error: {garch_standard_error:.6f}")
print(f"GARCH Z-score: {garch_z_score:.4f}")
print(f"Within 95% threshold: {abs(garch_z_score) <= 1.96}")

GARCH Standard Error: 0.086301
GARCH Z-score: 6.6685
Within 95% threshold: False


### Discretization Error: Numerical Methods versus Black–Scholes

Discretization error is evaluated using the absolute difference between each numerical pricing method and the analytical Black–Scholes benchmark. The errors are also normalized by the market half-spread to assess their economic significance.

These comparisons use the selected numerical grids. Convergence experiments conducted in the individual model notebooks provide additional evidence that the discrepancies arise from numerical approximation rather than from differences in volatility assumptions.

In [24]:
numerical_methods = reconciliation_df[
    reconciliation_df["Method"].isin([
        "CRR Binomial",
        "JRR Binomial",
        "Explicit FDM",
        "Fully Implicit FDM",
        "Crank-Nicolson FDM"
    ])
].copy()

numerical_methods["Absolute BS Error"] = (
    numerical_methods["Price"] - bs_call
).abs()

numerical_methods["Error in Half-Spreads"] = (
    numerical_methods["Absolute BS Error"] / half_spread
)

display(
    numerical_methods[
        [
            "Method",
            "Price",
            "Absolute BS Error",
            "Error in Half-Spreads"
        ]
    ]
)

,Method,Price,Absolute BS Error,Error in Half-Spreads
1,CRR Binomial,22.177026,0.002006,0.011465
2,JRR Binomial,22.161425,0.013594,0.077683
4,Explicit FDM,22.175143,0.000123,0.000705
5,Fully Implicit FDM,22.174623,0.000397,0.002267
6,Crank-Nicolson FDM,22.175025,0.000005,0.000031


### Quote Noise: Market Bid–Ask Consistency

Market quote uncertainty is assessed by comparing each model price with the observed bid–ask interval. A model price within the quoted spread is classified as consistent with the observed trading range, while a price outside the spread is flagged for further investigation.

This diagnostic does not establish that a model is correctly specified. The bid–ask interval reflects transaction costs and market conditions rather than a statistical confidence interval for the theoretical option value.

In [25]:
quote_df = reconciliation_df[
    [
        "Method",
        "Price",
        "Signed Distance (Half-Spreads)"
    ]
].copy()

quote_df["Within Bid-Ask"] = (
    (quote_df["Price"] >= market_bid)
    & (quote_df["Price"] <= market_ask)
)

quote_df["Distance Beyond Spread"] = (
    quote_df["Signed Distance (Half-Spreads)"].abs() - 1
).clip(lower=0)

display(quote_df)

,Method,Price,Signed Distance (Half-Spreads),Within Bid-Ask,Distance Beyond Spread
0,Black-Scholes,22.175020,0.000112,True,0.000000
1,CRR Binomial,22.177026,0.011577,True,0.000000
2,JRR Binomial,22.161425,-0.077570,True,0.000000
3,Monte Carlo,22.194452,0.111155,True,0.000000
4,Explicit FDM,22.175143,0.000817,True,0.000000
5,Fully Implicit FDM,22.174623,-0.002154,True,0.000000
6,Crank-Nicolson FDM,22.175025,0.000143,True,0.000000
7,GARCH Monte Carlo,22.750521,3.288692,False,2.288692


### Reconciliation and Model Risk: Final Assessment

**Input Uncertainty.** The Black–Scholes price is sensitive to the volatility assumption. Increasing or decreasing implied volatility by one percentage point changes the option value by approximately $0.427, equivalent to 2.44 market half-spreads. This demonstrates that uncertainty in volatility inputs can be economically more important than numerical pricing errors.

**Discretization Error.** The selected numerical methods produce relatively small deviations from the analytical Black–Scholes benchmark. The CRR and JRR binomial trees have absolute pricing errors of $0.002006 and $0.013594, respectively. The explicit, fully implicit, and Crank–Nicolson finite-difference schemes have errors of approximately $0.000123, $0.000397, and $0.000005. These errors are all below 0.08 market half-spreads. The separate grid-refinement experiments provide evidence of numerical convergence, although individual errors need not decrease monotonically because of error cancellation.

**Sampling Error.** The GBM Monte Carlo estimate of $22.194452 differs from the Black–Scholes benchmark by only 0.2664 estimated standard errors. Its 95% confidence interval contains the analytical price, indicating that the observed discrepancy is consistent with simulation sampling variability.

**Model Specification and Calibration Risk.** The GARCH Monte Carlo estimate is $22.750521, compared with the Black–Scholes price of $22.175020. The standardized difference is approximately 6.6685 GARCH simulation standard errors, and the GARCH 95% confidence interval excludes the Black–Scholes benchmark. Therefore, simulation sampling error alone is insufficient to explain the difference. The discrepancy warrants investigation of the volatility dynamics, calibration choices, and risk-neutral pricing assumptions. It does not, by itself, establish that either model is correctly specified.

**Quote Noise.** The observed market bid and ask are $22.00 and $22.35, giving a midquote of $22.175 and a half-spread of $0.175. Seven of the eight reported pricing methods produce estimates within the quoted interval. The GARCH Monte Carlo estimate lies above the ask by approximately $0.4005, or 2.2887 half-spreads. The quoted spread provides an economically relevant comparison range, but it is not a statistical confidence interval or proof of fundamental value.

### Assessment of the Four-Decimal-Place Agreement Statement

**The statement is false.**

Agreement between Black–Scholes, Monte Carlo, a binomial tree, and Crank–Nicolson to four decimal places would provide strong evidence of numerical consistency under a common set of assumptions. It would not establish that the market option has been correctly priced.

These methods may share the same implied volatility, risk-free rate, maturity, payoff specification, and underlying stochastic process. Consequently, all four methods could agree closely while inheriting the same incorrect input or model assumption.

The volatility sensitivity experiment illustrates this distinction: a one-percentage-point change in implied volatility changes the theoretical price by approximately $0.427, even though the numerical algorithms can still agree closely when evaluated using the revised input.

Furthermore, the GARCH Monte Carlo result demonstrates that changing the assumed volatility dynamics can generate a materially different option value. Market bid–ask spreads, liquidity, calibration uncertainty, and model risk must therefore be considered alongside numerical convergence.

**In conclusion, numerical agreement establishes computational consistency, not market pricing correctness.** A credible valuation requires both reliable numerical implementation and defensible economic assumptions, supported by market calibration and sensitivity analysis.

**Risk-Neutral Measure Specification in GARCH Pricing**

The relatively large discrepancy between the GARCH Monte Carlo price and the Black–Scholes benchmark may partly arise from the specification of the risk-neutral measure. GARCH parameters are typically estimated under the physical probability measure \(P\), whereas option valuation requires expectations under a risk-neutral measure \(Q\).

Unlike the constant-volatility Black–Scholes framework, the transformation from \(P\) to \(Q\) in a GARCH setting may affect the dynamics of conditional variance, not merely the expected return. Furthermore, different risk-neutralization approaches, such as Duan's locally risk-neutral valuation relationship or the Esscher transform, may produce different option values.

The observed discrepancy of approximately $0.576, equivalent to 6.67 estimated Monte Carlo standard errors, suggests that sampling error alone cannot explain the difference. The choice of risk-neutral measure is therefore a plausible source of model risk, although the present numerical evidence does not isolate it from other specification or calibration effects.

A useful robustness test would be to reprice the same option under alternative risk-neutral GARCH specifications while holding other inputs constant, and compare the results against market quotations across multiple strikes and maturities.